# Human Activity Recognition (UCI HAR)

The UCI HAR dataset has 30 volunteers wearing a waist-mounted smartphone, with the 3-axis accelerometer + gyroscope sampling at 50Hz. The dataset's authors have already done all the windowing + handcrafted feature extraction — each row in `X_train.txt` is a 561-dim feature vector summarizing a 2.56s window, labeled with one of 6 activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING).

A linear model on these features works embarrassingly well. RandomForest does a little better and is more honest about which features carry which classes. I'm not training a 1D-CNN on raw windows — that would be the next step if I cared about end-to-end.

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

DATA = Path("../../data/har_uci")  # expects the UCI HAR Dataset extracted here
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

needed = ["train/X_train.txt", "train/y_train.txt",
          "test/X_test.txt",  "test/y_test.txt",
          "activity_labels.txt"]
missing = [n for n in needed if not (DATA/n).exists()]
print("missing files:", missing if missing else "none")

missing files: ['train/X_train.txt', 'train/y_train.txt', 'test/X_test.txt', 'test/y_test.txt', 'activity_labels.txt']


In [2]:
if not missing:
    Xtr = pd.read_csv(DATA/"train/X_train.txt", delim_whitespace=True, header=None).values
    ytr = pd.read_csv(DATA/"train/y_train.txt", header=None).values.ravel()
    Xte = pd.read_csv(DATA/"test/X_test.txt",  delim_whitespace=True, header=None).values
    yte = pd.read_csv(DATA/"test/y_test.txt",  header=None).values.ravel()
    labels = pd.read_csv(DATA/"activity_labels.txt", sep=" ", header=None, names=["id","activity"])
    label_map = dict(zip(labels.id, labels.activity))
    print(f"Xtr={Xtr.shape}  Xte={Xte.shape}")
    print(pd.Series([label_map[i] for i in ytr]).value_counts())
else:
    print("download from https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones")
    Xtr = ytr = Xte = yte = label_map = None

download from https://archive.ics.uci.edu/ml/datasets/Human+Activity+Recognition+Using+Smartphones


## Quick baseline + RF

In [3]:
if Xtr is not None:
    lr = LogisticRegression(max_iter=2000, C=0.1)
    lr.fit(Xtr, ytr)
    acc_lr = accuracy_score(yte, lr.predict(Xte))
    print(f"LogReg test accuracy: {acc_lr:.4f}")

    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=7)
    rf.fit(Xtr, ytr)
    yhat = rf.predict(Xte)
    acc_rf = accuracy_score(yte, yhat)
    print(f"RF     test accuracy: {acc_rf:.4f}")
    print()
    target_names = [label_map[i] for i in sorted(label_map)]
    print(classification_report(yte, yhat, target_names=target_names))
else:
    print("skipped — load the dataset first")

skipped — load the dataset first


In [4]:
if Xtr is not None:
    cm = confusion_matrix(yte, yhat)
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(target_names))); ax.set_xticklabels(target_names, rotation=45, ha="right")
    ax.set_yticks(range(len(target_names))); ax.set_yticklabels(target_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i,j], ha="center", va="center",
                    color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=8)
    ax.set_title("RF confusion matrix")
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(ART/"har_cm.png", dpi=110); plt.show()

## Notes

The confusion is concentrated where you'd expect — `SITTING` vs `STANDING`. Both are stationary, and the only signal distinguishing them is the body's tilt as captured by gravity components in the accelerometer. Walking vs walking-upstairs vs walking-downstairs is also a little fuzzy, but the model gets most of those right.

The dataset is small, the splits are subject-disjoint (good), and the features are heavily engineered (less good if you want to argue your model "learned" anything). A 1D-CNN on the raw inertial signals would be the proper sequel here — and a fair test of whether deep learning earns its keep on a sensor dataset this size. My honest guess: a few percent better at most.